# 02 — Paper figures

This notebook produces every figure in the paper.

**Two modes**, switched automatically:

1. **Real model data** — if `outputs/evaluation/embeddings.npz` and
   `outputs/evaluation/predictions.npz` exist (i.e. you have run
   `scripts/evaluate.py` on a trained checkpoint), every figure is
   computed from those real arrays.

2. **Reported-paper numbers** — if no `.npz` is present, the notebook
   falls back to the numbers reported in Tables I–III and Sec. V of
   the paper, and prints a clear banner above each figure so reviewers
   can tell at a glance.

Either way, all generated PDFs go to `outputs/figures/`.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

EVAL_DIR = ROOT / 'outputs/evaluation'
FIG_DIR  = ROOT / 'outputs/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

HAVE_REAL = (EVAL_DIR / 'embeddings.npz').exists() and (EVAL_DIR / 'predictions.npz').exists()
print(f'Real evaluation outputs found: {HAVE_REAL}')
print(f'Figure output directory:        {FIG_DIR}')

## Section 1 — Reported-paper-number figures

These reproduce **Table I (main results), Table II (ablation), Table III (backbones)**, and the
**temperature / noise / per-signer** plots from Sec. V. They always render regardless of whether
you have a trained checkpoint, because they only use the numerical values reported in the paper.

In [ ]:
from src.utils.visualization import ReportedFigures
rf = ReportedFigures()
rf.generate_all(FIG_DIR)
print('\nGenerated PDFs:')
for f in sorted(FIG_DIR.glob('fig_*.pdf')):
    print(' ', f.relative_to(ROOT))

In [ ]:
from IPython.display import Image, display
# Quick inline preview of the PNG versions
for name in ['fig_table1_top1', 'fig_table2_ablation', 'fig_table3_backbones',
             'fig_temperature_sensitivity', 'fig_noise_sensitivity',
             'fig_per_signer_reported']:
    png = FIG_DIR / f'{name}.png'
    if png.exists():
        display(Image(filename=str(png)))

## Section 2 — Real-data figures (require a trained checkpoint)

These read from `outputs/evaluation/embeddings.npz` and `outputs/evaluation/predictions.npz`.
If the files are missing, this section is skipped with a hint about how to generate them.

In [ ]:
if not HAVE_REAL:
    print('No real evaluation outputs found.\n\n'
          'To generate them, train a model and run:\n\n'
          '   python scripts/evaluate.py \\\n'
          '       --checkpoint outputs/<exp>/checkpoints/best.pt \\\n'
          '       --video_dir data/azsld/videos \\\n'
          '       --skeleton_dir data/azsld/skeletons \\\n'
          '       --descriptions data/azsld/descriptions.json \\\n'
          '       --unseen_glosses data/azsld/splits/unseen_glosses.txt \\\n'
          '       --signer_map data/azsld/splits/signer_map.json \\\n'
          '       --output_dir outputs/evaluation\n')
else:
    from src.utils.visualization import FigureGenerator
    signer_map_path = ROOT / 'data/azsld/splits/signer_map.json'
    fg = FigureGenerator.from_npz(
        eval_dir=EVAL_DIR,
        signer_map_path=str(signer_map_path) if signer_map_path.exists() else None,
        output_dir=FIG_DIR,
    )
    fg.generate_all()
    print('\nGenerated real-data PDFs:')
    for f in sorted(FIG_DIR.glob('fig[A-H]_*.pdf')):
        print(' ', f.relative_to(ROOT))

## Section 3 — Custom analysis

The cell below loads the raw arrays so you can experiment freely
(e.g. inspect particular wrong predictions, slice by signer, etc).
Only runs in real-data mode.

In [ ]:
if HAVE_REAL:
    import numpy as np
    emb  = np.load(EVAL_DIR / 'embeddings.npz', allow_pickle=True)
    pred = np.load(EVAL_DIR / 'predictions.npz', allow_pickle=True)
    metrics = json.loads((EVAL_DIR / 'metrics.json').read_text())
    print('vis_embeds   :', emb['visual'].shape)
    print('text_embeds  :', emb['text'].shape)
    print('similarities :', pred['similarities'].shape)
    print('top-1 acc    :', metrics.get('top1'))
    print('top-5 acc    :', metrics.get('top5'))
    print('mAP          :', metrics.get('mAP'))
    
    # Inspect 10 hardest mistakes
    wrong = np.where(pred['predictions'] != pred['targets'])[0]
    print(f'\nWrong samples: {len(wrong)}/{len(pred["targets"])}')
    for i in wrong[:10]:
        gt   = emb['class_names'][pred['targets'][i]]
        pr   = emb['class_names'][pred['predictions'][i]]
        sims = pred['similarities'][i]
        margin = sims[pred['predictions'][i]] - sims[pred['targets'][i]]
        print(f'  vid={pred["video_ids"][i]:<24}  gt={gt:<10}  pred={pr:<10}  margin={margin:+.3f}')
else:
    print('Run evaluate.py first to populate outputs/evaluation/.')